# Semantic Retrieval

This notebook builds the retrieval layer of the protocol copilot.

We will load the validated chunks from Day 2, add deterministic provenance information, convert the chunk text into embeddings, store the embeddings in Chroma, and test semantic retrieval using realistic protocol questions.

The main goal is to verify that relevant evidence can be found before connecting retrieval to an LLM.

## 1. Load the retrieval-ready chunks

Day 2 produced a validated dataset of 172 chunks from five protocol documents.

We will load that saved dataset and verify the basic structure before creating embeddings. This keeps Day 3 independent from the PDF extraction and chunking notebooks.

In [1]:
from pathlib import Path
import pandas as pd


# Load the final chunk dataset created on Day 2
chunks_path = Path("../data/processed/protocol_chunks.jsonl")

chunks_df = pd.read_json(
    chunks_path,
    lines=True,
)


print("Dataset shape:", chunks_df.shape)
print("Documents:", chunks_df["document_id"].nunique())

source_pages = (
    chunks_df[
        ["document_id", "page_number"]
    ]
    .drop_duplicates()
)

print("Source pages:", len(source_pages))
print("Chunks:", len(chunks_df))

print("\nColumns:")
print(chunks_df.columns.tolist())

print("\nFirst three chunk IDs:")
print(chunks_df["chunk_id"].head(3))

Dataset shape: (172, 8)
Documents: 5
Source pages: 52
Chunks: 172

Columns:
['chunk_id', 'document_id', 'document_name', 'filename', 'page_number', 'chunk_index', 'text', 'token_count']

First three chunk IDs:
0    PROTO_001_P001_C001
1    PROTO_001_P001_C002
2    PROTO_001_P001_C003
Name: chunk_id, dtype: str


## 2. Add deterministic chunk fingerprints

Each chunk already has a stable ID that records its document, page and chunk position.

We will also create a SHA-256 hash of the chunk text. This acts as a deterministic fingerprint: the same text always produces the same hash, while a change to the text produces a different hash.

This gives us an additional integrity check for the evidence that will later be indexed and retrieved.

In [2]:
import hashlib


# Create a deterministic fingerprint for each chunk's exact text
chunks_df["chunk_hash"] = chunks_df["text"].apply(
    lambda text: hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()
)


# Record the preprocessing and chunking strategy in readable metadata
chunks_df["preprocessing_version"] = "clean_v1"
chunks_df["chunking_config"] = "recursive_500_75_page_contained"


print("Chunks:", len(chunks_df))
print(
    "Unique chunk hashes:",
    chunks_df["chunk_hash"].nunique(),
)

print(
    "Missing chunk hashes:",
    chunks_df["chunk_hash"].isna().sum(),
)

print("\nExample:")
print(
    chunks_df[
        [
            "chunk_id",
            "chunk_hash",
            "preprocessing_version",
            "chunking_config",
        ]
    ].head(3)
)

Chunks: 172
Unique chunk hashes: 172
Missing chunk hashes: 0

Example:
              chunk_id                                         chunk_hash  \
0  PROTO_001_P001_C001  6785bfa66902582e38f723e632eacfc804f10bb4eb386c...   
1  PROTO_001_P001_C002  fdd9ffba00d6d605e683eef75846cd4932bab02c22861d...   
2  PROTO_001_P001_C003  8f38bd3e7797b69a3e39849a03c71c30bf24a480d47d4a...   

  preprocessing_version                  chunking_config  
0              clean_v1  recursive_500_75_page_contained  
1              clean_v1  recursive_500_75_page_contained  
2              clean_v1  recursive_500_75_page_contained  


## 4. Test semantic similarity

Before embedding the full protocol corpus, we will test the embedding model on a few simple sentences.

An embedding converts text into a numerical vector that represents its meaning. Sentences with similar meanings should have more similar vectors, even when they use different words.

This small test helps us understand semantic retrieval before applying it to all 172 protocol chunks.

In [3]:
from sentence_transformers import SentenceTransformer, util


# Load a small pretrained embedding model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(
    embedding_model_name
)


# Three example sentences
sentences = [
    "Participants must be at least 18 years old to enrol.",
    "What is the minimum age required for study participation?",
    "The study measures anxiety using a questionnaire.",
]


# Convert each sentence into an embedding vector
sentence_embeddings = embedding_model.encode(
    sentences,
    convert_to_tensor=True,
)


# Compare every sentence with every other sentence
similarity_matrix = util.cos_sim(
    sentence_embeddings,
    sentence_embeddings,
)


print("Embedding model:", embedding_model_name)
print("Number of sentences:", len(sentences))
print(
    "Embedding dimensions:",
    sentence_embeddings.shape[1],
)


print("\nSentences:")
for index, sentence in enumerate(sentences, start=1):
    print(f"{index}. {sentence}")


print("\nCosine similarity matrix:")
print(
    similarity_matrix
    .cpu()
    .numpy()
    .round(3)
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Number of sentences: 3
Embedding dimensions: 384

Sentences:
1. Participants must be at least 18 years old to enrol.
2. What is the minimum age required for study participation?
3. The study measures anxiety using a questionnaire.

Cosine similarity matrix:
[[1.    0.739 0.113]
 [0.739 1.    0.172]
 [0.113 0.172 1.   ]]


## 5. Create embeddings for the protocol chunks

The test sentences showed that the embedding model can recognise similar meanings even when the wording is different.

We will now convert all 172 protocol chunks into embedding vectors. Each chunk will receive a 384-dimensional numerical representation that can later be stored and searched using Chroma.

At this stage we are creating the searchable numerical representation of the corpus, not generating any answers with an LLM.

In [4]:
# Convert all final protocol chunks into embedding vectors
chunk_texts = chunks_df["text"].tolist()

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)


print("Chunks embedded:", len(chunk_texts))
print("Embedding matrix shape:", chunk_embeddings.shape)

print(
    "Embedding dimensions per chunk:",
    chunk_embeddings.shape[1],
)

print(
    "Missing values in embeddings:",
    pd.isna(chunk_embeddings).sum(),
)


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Chunks embedded: 172
Embedding matrix shape: (172, 384)
Embedding dimensions per chunk: 384
Missing values in embeddings: 0


## 6. Build the Chroma vector index

The 172 protocol chunks now have embedding vectors that represent their meaning.

We will store each chunk, its embedding and its provenance metadata in Chroma. Chroma will act as the vector database used for semantic retrieval.

The database will be stored on disk so that it can be reused later without rebuilding the index every time the application starts.

In [5]:
import chromadb


# Choose where the persistent Chroma database will be stored
chroma_path = Path("../data/chroma")


# Create a persistent Chroma client
client = chromadb.PersistentClient(
    path=str(chroma_path)
)


# Create or reuse the collection for our protocol chunks
collection = client.get_or_create_collection(
    name="protocol_chunks_v1",
    metadata={
        "embedding_model": embedding_model_name,
        "chunking_config": "recursive_500_75_page_contained",
        "preprocessing_version": "clean_v1",
    },
)


# Prepare the chunk metadata that will be stored with each vector
chunk_metadatas = []

for _, row in chunks_df.iterrows():
    chunk_metadatas.append(
        {
            "document_id": row["document_id"],
            "document_name": row["document_name"],
            "filename": row["filename"],
            "page_number": int(row["page_number"]),
            "chunk_index": int(row["chunk_index"]),
            "token_count": int(row["token_count"]),
            "chunk_hash": row["chunk_hash"],
            "preprocessing_version": row["preprocessing_version"],
            "chunking_config": row["chunking_config"],
        }
    )


# Store the chunks, vectors and metadata in Chroma
# upsert makes this cell safe to rerun with the same chunk IDs
collection.upsert(
    ids=chunks_df["chunk_id"].tolist(),
    documents=chunks_df["text"].tolist(),
    embeddings=chunk_embeddings.tolist(),
    metadatas=chunk_metadatas,
)


print("Collection name:", collection.name)
print("Records stored:", collection.count())
print("Database path:", chroma_path)

Collection name: protocol_chunks_v1
Records stored: 172
Database path: ../data/chroma


## 7. Test semantic retrieval on the protocol corpus

The protocol chunks are now stored in Chroma together with their embeddings and provenance metadata.

We will test the retrieval system using a real question about the INTEGRA protocol. The question will first be converted into an embedding using the same model as the document chunks. Chroma will then search all 172 stored chunks and return the closest matches.

At this stage we are evaluating retrieval only. No LLM is generating an answer yet.

In [6]:
# Ask a question about information contained in the protocol corpus
query_text = (
    "What are the follow-up time points "
    "for participants in the INTEGRA study?"
)


# Convert the question into the same 384-dimensional embedding format
query_embedding = embedding_model.encode(
    query_text,
    convert_to_numpy=True,
    normalize_embeddings=True,
)


# Search the 172 stored protocol chunks
results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    include=[
        "documents",
        "metadatas",
        "distances",
    ],
)


print("Question:")
print(query_text)

print("\nTop retrieved chunks:")
print("=" * 70)


# Display the five closest results
for rank in range(len(results["ids"][0])):
    chunk_id = results["ids"][0][rank]
    metadata = results["metadatas"][0][rank]
    distance = results["distances"][0][rank]
    text = results["documents"][0][rank]

    print(f"\nRank {rank + 1}")
    print("Chunk ID:", chunk_id)
    print("Document:", metadata["document_name"])
    print("Page:", metadata["page_number"])
    print("Distance:", round(distance, 4))

    # Show a short preview of the retrieved evidence
    print("Text:")
    print(text[:500].replace("\n", " "))

    print("\n" + "-" * 70)

Question:
What are the follow-up time points for participants in the INTEGRA study?

Top retrieved chunks:

Rank 1
Chunk ID: PROTO_003_P002_C002
Document: INTEGRA
Page: 2
Distance: 1.1827
Text:
average of 2.9 years, with values of > 7% before change, when switching from monotherapy to combination ther- apy was observed. Additionally, in an assessment of the GEDAPS group carried out in Catalonia, clinical inertia was detected in 33% of patients, and the mean HbA1c level to perform the treatment change was 8.4%. [ 8]. Sometimes, clinical inertia is related to the non-com- pliance of the patient [ 3, 9]. As seen in the Grant et al. [10] study, interventions oriented to improve patient ad-

----------------------------------------------------------------------

Rank 2
Chunk ID: PROTO_001_P003_C001
Document: CERTAIN
Page: 3
Distance: 1.1991
Text:
3 Panda R, et al. BMJ Open 2022;12:e048628. doi:10.1136/bmjopen-2021-048628 Open access Study design The study will be conducted in two different 

## 8. Test retrieval with document metadata filtering

The first corpus-wide search did not retrieve the expected follow-up evidence reliably.

The question specifies the INTEGRA study, and the source document is already stored as metadata. Rather than asking semantic similarity to identify both the document and the relevant passage, we can filter the search to INTEGRA first and then rank only its chunks by semantic similarity.

This separates document selection from evidence retrieval and gives us a cleaner test of whether the embedding model can find the correct passage within the intended protocol.

In [7]:
# Search only within the INTEGRA protocol
filtered_results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=5,
    where={
        "document_name": "INTEGRA"
    },
    include=[
        "documents",
        "metadatas",
        "distances",
    ],
)


print("Question:")
print(query_text)

print("\nDocument filter: INTEGRA")

print("\nTop retrieved chunks:")
print("=" * 70)


for rank in range(len(filtered_results["ids"][0])):
    chunk_id = filtered_results["ids"][0][rank]
    metadata = filtered_results["metadatas"][0][rank]
    distance = filtered_results["distances"][0][rank]
    text = filtered_results["documents"][0][rank]

    print(f"\nRank {rank + 1}")
    print("Chunk ID:", chunk_id)
    print("Page:", metadata["page_number"])
    print("Distance:", round(distance, 4))

    # Keep the preview short so VS Code does not truncate the output
    preview = text.replace("\n", " ")[:350]

    print("Text:")
    print(preview)

    print("\n" + "-" * 70)

Question:
What are the follow-up time points for participants in the INTEGRA study?

Document filter: INTEGRA

Top retrieved chunks:

Rank 1
Chunk ID: PROTO_003_P002_C002
Page: 2
Distance: 1.1827
Text:
average of 2.9 years, with values of > 7% before change, when switching from monotherapy to combination ther- apy was observed. Additionally, in an assessment of the GEDAPS group carried out in Catalonia, clinical inertia was detected in 33% of patients, and the mean HbA1c level to perform the treatment change was 8.4%. [ 8]. Sometimes, clinical in

----------------------------------------------------------------------

Rank 2
Chunk ID: PROTO_003_P008_C002
Page: 8
Distance: 1.263
Text:
According to a study performed by our research group [16], we estimated that each GP would have 15 potential candidate patients with HbA1c > 9%. Assuming that the intra-class correlation coefficient for Primary Care is 0.05 [ 29], resulting in a 1.7 design effect, the required sample size will be 42 * 1.7 

## 9. Check the embedding model input limit

The filtered search still did not rank the expected INTEGRA follow-up evidence highly.

Before changing the retrieval strategy, we need to check whether the embedding model can process the full length of our approximately 500-token chunks. If the model has a smaller maximum input length, longer chunks may be truncated when their embeddings are created.

This would mean some information in a chunk is stored as text in Chroma but is not fully represented by its embedding.

In [8]:
# Inspect how much text the embedding model can process at once
print("Embedding model:", embedding_model_name)
print("Model maximum sequence length:", embedding_model.max_seq_length)

print("\nFinal chunk token statistics:")
print(
    chunks_df["token_count"]
    .describe()
    .round(1)
)

# Count chunks that are longer than the model's reported sequence limit
chunks_over_model_limit = (
    chunks_df["token_count"] > embedding_model.max_seq_length
).sum()

print(
    "\nChunks longer than the model sequence limit:",
    chunks_over_model_limit,
)

print(
    "Total chunks:",
    len(chunks_df),
)

Embedding model: sentence-transformers/all-MiniLM-L6-v2
Model maximum sequence length: 256

Final chunk token statistics:
count    172.0
mean     399.8
std      112.2
min       81.0
25%      365.8
50%      454.5
75%      465.0
max      496.0
Name: token_count, dtype: float64

Chunks longer than the model sequence limit: 148
Total chunks: 172


## 10. Measure chunk lengths using the embedding model tokenizer

The embedding model has a maximum sequence length of 256 tokens, while our Day 2 chunk sizes were measured using a different tokenizer.

To determine whether truncation is actually occurring, we will measure every chunk using MiniLM's own tokenizer. This gives us an exact comparison between the chunk lengths and the embedding model's input limit.

In [9]:
# Measure every chunk using MiniLM's own tokenizer
def get_embedding_token_count(text):
    encoded = embedding_model.tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )

    return len(encoded["input_ids"])


# Store the MiniLM token count separately
chunks_df["embedding_token_count"] = chunks_df["text"].apply(
    get_embedding_token_count
)


# Check how many chunks exceed MiniLM's 256-token limit
chunks_over_embedding_limit = (
    chunks_df["embedding_token_count"]
    > embedding_model.max_seq_length
)

print("Embedding model maximum sequence length:",
      embedding_model.max_seq_length)

print("\nMiniLM token statistics:")
print(
    chunks_df["embedding_token_count"]
    .describe()
    .round(1)
)

print(
    "\nChunks above MiniLM limit:",
    chunks_over_embedding_limit.sum()
)

print(
    "Chunks within MiniLM limit:",
    (~chunks_over_embedding_limit).sum()
)

print(
    "Percentage above limit:",
    round(
        chunks_over_embedding_limit.mean() * 100,
        1
    ),
    "%"
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (401 > 256). Running this sequence through the model will result in indexing errors


Embedding model maximum sequence length: 256

MiniLM token statistics:
count    172.0
mean     376.7
std      105.7
min       69.0
25%      355.8
50%      422.5
75%      442.0
max      502.0
Name: embedding_token_count, dtype: float64

Chunks above MiniLM limit: 145
Chunks within MiniLM limit: 27
Percentage above limit: 84.3 %


## 11. Test a longer-context embedding model

The MiniLM baseline revealed a mismatch between the embedding model and our chunking strategy. MiniLM accepts up to 256 tokens, while 84.3% of our chunks exceed that limit using MiniLM's own tokenizer.

Rather than immediately changing the chunking strategy, we will test an embedding model with a longer input limit. The goal is to determine whether our existing 500-token chunks can be represented without substantial truncation.

In [10]:
# Load a longer-context embedding model
candidate_model_name = "BAAI/bge-small-en-v1.5"

candidate_model = SentenceTransformer(
    candidate_model_name
)


print("Candidate model:", candidate_model_name)
print(
    "Maximum sequence length:",
    candidate_model.max_seq_length,
)
print(
    "Embedding dimensions:",
    candidate_model.get_sentence_embedding_dimension(),
)


# Measure each chunk using the candidate model's own tokenizer
def get_candidate_token_count(text):
    encoded = candidate_model.tokenizer(
        text,
        add_special_tokens=True,
        truncation=False,
    )

    return len(encoded["input_ids"])


chunks_df["candidate_token_count"] = chunks_df["text"].apply(
    get_candidate_token_count
)


chunks_over_candidate_limit = (
    chunks_df["candidate_token_count"]
    > candidate_model.max_seq_length
)


print("\nCandidate token statistics:")
print(
    chunks_df["candidate_token_count"]
    .describe()
    .round(1)
)

print(
    "\nChunks above candidate limit:",
    chunks_over_candidate_limit.sum(),
)

print(
    "Chunks within candidate limit:",
    (~chunks_over_candidate_limit).sum(),
)

print(
    "Percentage above limit:",
    round(
        chunks_over_candidate_limit.mean() * 100,
        1,
    ),
    "%",
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Candidate model: BAAI/bge-small-en-v1.5
Maximum sequence length: 512
Embedding dimensions: 384

Candidate token statistics:
count    172.0
mean     376.7
std      105.7
min       69.0
25%      355.8
50%      422.5
75%      442.0
max      502.0
Name: candidate_token_count, dtype: float64

Chunks above candidate limit: 0
Chunks within candidate limit: 172
Percentage above limit: 0.0 %


/var/folders/j9/c4fkmh8s679bzbcy5bdmsq300000gn/T/ipykernel_30013/3170017220.py:16: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  candidate_model.get_sentence_embedding_dimension(),


## 12. Create embeddings with the selected model

The MiniLM baseline was not compatible with our chunk sizes because most chunks exceeded its 256-token input limit.

The BGE candidate supports up to 512 tokens, and all 172 chunks fit within that limit. We will therefore use BGE as the embedding model for the retrieval system and recreate the corpus embeddings without truncation.

In [11]:
# Use BGE as the selected embedding model
embedding_model_name = candidate_model_name
embedding_model = candidate_model


# Convert all 172 protocol chunks using the selected BGE model
chunk_embeddings_bge = embedding_model.encode(
    chunks_df["text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)


print("Selected embedding model:", embedding_model_name)
print("Chunks embedded:", len(chunk_embeddings_bge))
print("Embedding matrix shape:", chunk_embeddings_bge.shape)
print("Embedding dimensions:", chunk_embeddings_bge.shape[1])

print(
    "Missing values:",
    pd.isna(chunk_embeddings_bge).sum(),
)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Selected embedding model: BAAI/bge-small-en-v1.5
Chunks embedded: 172
Embedding matrix shape: (172, 384)
Embedding dimensions: 384
Missing values: 0


## 13. Build the BGE vector index

All 172 chunks fit within the selected BGE embedding model's input limit and have now been embedded successfully.

We will create a separate Chroma collection for these BGE embeddings. The earlier MiniLM collection will remain as a baseline experiment, while this new collection will be used for the improved retrieval test.

In [12]:
# Create a separate collection for the selected BGE embeddings
bge_collection = client.get_or_create_collection(
    name="protocol_chunks_bge_v1",
    metadata={
        "embedding_model": embedding_model_name,
        "chunking_config": "recursive_500_75_page_contained",
        "preprocessing_version": "clean_v1",
    },
)


# Prepare the metadata stored alongside every chunk
bge_metadatas = []

for _, row in chunks_df.iterrows():
    bge_metadatas.append(
        {
            "document_id": row["document_id"],
            "document_name": row["document_name"],
            "filename": row["filename"],
            "page_number": int(row["page_number"]),
            "chunk_index": int(row["chunk_index"]),
            "token_count": int(row["token_count"]),
            "chunk_hash": row["chunk_hash"],
            "preprocessing_version": row["preprocessing_version"],
            "chunking_config": row["chunking_config"],
        }
    )


# Store the BGE vectors, chunk text and provenance metadata
bge_collection.upsert(
    ids=chunks_df["chunk_id"].tolist(),
    documents=chunks_df["text"].tolist(),
    embeddings=chunk_embeddings_bge.tolist(),
    metadatas=bge_metadatas,
)


print("Collection name:", bge_collection.name)
print("Records stored:", bge_collection.count())
print("Embedding model:", embedding_model_name)
print("Database path:", chroma_path)

Collection name: protocol_chunks_bge_v1
Records stored: 172
Embedding model: BAAI/bge-small-en-v1.5
Database path: ../data/chroma


## 14. Retest retrieval with the selected BGE model

The original MiniLM retrieval struggled to find the expected INTEGRA follow-up evidence, and we discovered that most chunks exceeded MiniLM's input limit.

We have now rebuilt the vector index using BGE, which can represent all 172 chunks without truncation.

We will repeat the same INTEGRA question and inspect whether the relevant follow-up evidence is ranked more highly.

In [13]:
# Use the exact same question as before
query_text = (
    "What are the follow-up time points "
    "for participants in the INTEGRA study?"
)


# BGE works well when the retrieval purpose is stated in the query
bge_query_text = (
    "Represent this sentence for searching relevant passages: "
    + query_text
)


# Convert the question into a BGE embedding
bge_query_embedding = embedding_model.encode(
    bge_query_text,
    convert_to_numpy=True,
    normalize_embeddings=True,
)


# Search only within the INTEGRA protocol
bge_results = bge_collection.query(
    query_embeddings=[bge_query_embedding.tolist()],
    n_results=5,
    where={
        "document_name": "INTEGRA"
    },
    include=[
        "documents",
        "metadatas",
        "distances",
    ],
)


print("Question:")
print(query_text)

print("\nDocument filter: INTEGRA")

print("\nTop retrieved chunks:")
print("=" * 70)


for rank in range(len(bge_results["ids"][0])):
    chunk_id = bge_results["ids"][0][rank]
    metadata = bge_results["metadatas"][0][rank]
    distance = bge_results["distances"][0][rank]
    text = bge_results["documents"][0][rank]

    print(f"\nRank {rank + 1}")
    print("Chunk ID:", chunk_id)
    print("Page:", metadata["page_number"])
    print("Distance:", round(distance, 4))

    # If "follow" appears in the chunk, show the text around it
    follow_position = text.lower().find("follow")

    if follow_position != -1:
        start = max(0, follow_position - 150)
        end = min(len(text), follow_position + 350)
        preview = text[start:end]
    else:
        preview = text[:400]

    print("Text:")
    print(preview.replace("\n", " "))

    print("\n" + "-" * 70)

Question:
What are the follow-up time points for participants in the INTEGRA study?

Document filter: INTEGRA

Top retrieved chunks:

Rank 1
Chunk ID: PROTO_003_P001_C001
Page: 1
Distance: 0.5747
Text:
S T U D Y P R O T O C O L Open Access INTEGRA study protocol: primary care intervention in type 2 diabetes patients with poor glycaemic control Àngels Molló 1,2,3† , Anna Berenguera 4,5† , Esther Rubinat 6,7,8,9, Bogdan Vlacho 10, Manel Mata 11,12,13,14,15, Josep Franch 16, Bonaventura Bolíbar 17,18 and Dídac Mauricio 19,20,21* Abstract Background: The management of hyperglycaemia and associated c

----------------------------------------------------------------------

Rank 2
Chunk ID: PROTO_003_P004_C001
Page: 4
Distance: 0.618
Text:
 potential participants. Sampling The sampling for the study was opportunistic [15]. Although not a theoretical sample, it was taken into account the following variables: gender, age, years of progression of T2DM, and type of treatment (oral anti- diabetics

## 15. Test OpenAI embeddings

The local BGE model provides our current retrieval baseline. We will now test OpenAI's `text-embedding-3-small` model on one real protocol chunk before embedding the full corpus.

This confirms that the API connection works and lets us inspect the embedding output before creating a separate OpenAI vector index.

In [18]:
from openai import OpenAI


# Create the OpenAI API client using the key loaded from .env
openai_client = OpenAI(
    api_key=api_key
)


# Select the OpenAI embedding model for our comparison
openai_embedding_model = "text-embedding-3-small"


# Use one real protocol chunk for the connection test
test_chunk_id = chunks_df.iloc[0]["chunk_id"]
test_chunk_text = chunks_df.iloc[0]["text"]


# Request an embedding for the chunk
response = openai_client.embeddings.create(
    model=openai_embedding_model,
    input=test_chunk_text,
)


# Extract the numerical embedding vector
test_openai_embedding = response.data[0].embedding


print("Embedding model:", openai_embedding_model)
print("Input chunk:", test_chunk_id)
print("Embedding dimensions:", len(test_openai_embedding))
print(
    "Embedding created successfully:",
    len(test_openai_embedding) > 0,
)

Embedding model: text-embedding-3-small
Input chunk: PROTO_001_P001_C001
Embedding dimensions: 1536
Embedding created successfully: True


## 16. Create OpenAI embeddings for the full corpus

The OpenAI connection test successfully created a 1536-dimensional embedding for one real protocol chunk.

We will now embed all 172 chunks using `text-embedding-3-small`. These embeddings will be stored in a separate Chroma collection so that OpenAI retrieval can be compared fairly against the local BGE baseline using the same documents, chunks and questions.

In [19]:
import numpy as np


# Collect the text from all 172 final chunks
openai_chunk_texts = chunks_df["text"].tolist()


# Request embeddings for the complete corpus
openai_response = openai_client.embeddings.create(
    model=openai_embedding_model,
    input=openai_chunk_texts,
)


# Sort by the returned input index to preserve chunk order
openai_embedding_records = sorted(
    openai_response.data,
    key=lambda item: item.index,
)


# Convert the returned embeddings into a numerical matrix
openai_chunk_embeddings = np.array(
    [
        item.embedding
        for item in openai_embedding_records
    ],
    dtype=np.float32,
)


print("Embedding model:", openai_embedding_model)
print("Chunks submitted:", len(openai_chunk_texts))
print("Embeddings returned:", len(openai_embedding_records))
print("Embedding matrix shape:", openai_chunk_embeddings.shape)
print(
    "Missing values:",
    np.isnan(openai_chunk_embeddings).sum(),
)

Embedding model: text-embedding-3-small
Chunks submitted: 172
Embeddings returned: 172
Embedding matrix shape: (172, 1536)
Missing values: 0


## 17. Build the OpenAI vector index

All 172 protocol chunks have now been embedded successfully using OpenAI's `text-embedding-3-small` model.

We will store these embeddings in a separate Chroma collection. Keeping the OpenAI and BGE indexes separate allows us to compare retrieval performance using exactly the same source chunks and questions.

In [20]:
# Create a separate Chroma collection for OpenAI embeddings
openai_collection = client.get_or_create_collection(
    name="protocol_chunks_openai_v1",
    metadata={
        "embedding_model": openai_embedding_model,
        "chunking_config": "recursive_500_75_page_contained",
        "preprocessing_version": "clean_v1",
    },
)


# The chunk metadata is identical to the metadata used for BGE
openai_metadatas = bge_metadatas


# Store OpenAI vectors, chunk text and provenance metadata
openai_collection.upsert(
    ids=chunks_df["chunk_id"].tolist(),
    documents=chunks_df["text"].tolist(),
    embeddings=openai_chunk_embeddings.tolist(),
    metadatas=openai_metadatas,
)


print("Collection name:", openai_collection.name)
print("Records stored:", openai_collection.count())
print("Embedding model:", openai_embedding_model)
print("Embedding dimensions:", openai_chunk_embeddings.shape[1])
print("Database path:", chroma_path)

Collection name: protocol_chunks_openai_v1
Records stored: 172
Embedding model: text-embedding-3-small
Embedding dimensions: 1536
Database path: ../data/chroma


## 18. Test retrieval with OpenAI embeddings

The same 172 protocol chunks are now indexed using both BGE and OpenAI embeddings.

We will repeat the INTEGRA follow-up question using the OpenAI vector index. The document filter and retrieval depth remain unchanged so that the result can be compared fairly with the BGE baseline.

The goal is to see which embedding model ranks the known follow-up evidence more effectively.

In [21]:
# Use the same question tested with the BGE index
openai_query_text = (
    "What are the follow-up time points "
    "for participants in the INTEGRA study?"
)


# Create an OpenAI embedding for the question
openai_query_response = openai_client.embeddings.create(
    model=openai_embedding_model,
    input=openai_query_text,
)

openai_query_embedding = (
    openai_query_response
    .data[0]
    .embedding
)


# Search only within the INTEGRA document,
# exactly as we did for the BGE comparison
openai_results = openai_collection.query(
    query_embeddings=[openai_query_embedding],
    n_results=5,
    where={
        "document_name": "INTEGRA"
    },
    include=[
        "documents",
        "metadatas",
        "distances",
    ],
)


print("Question:")
print(openai_query_text)

print("\nDocument filter: INTEGRA")

print("\nTop retrieved chunks:")
print("=" * 70)


for rank in range(len(openai_results["ids"][0])):
    chunk_id = openai_results["ids"][0][rank]
    metadata = openai_results["metadatas"][0][rank]
    distance = openai_results["distances"][0][rank]
    text = openai_results["documents"][0][rank]

    print(f"\nRank {rank + 1}")
    print("Chunk ID:", chunk_id)
    print("Page:", metadata["page_number"])
    print("Distance:", round(distance, 4))

    # If follow-up wording appears, show the surrounding evidence
    follow_position = text.lower().find("follow")

    if follow_position != -1:
        start = max(0, follow_position - 120)
        end = min(len(text), follow_position + 320)
        preview = text[start:end]
    else:
        preview = text[:320]

    print("Text:")
    print(preview.replace("\n", " "))

    print("\n" + "-" * 70)

Question:
What are the follow-up time points for participants in the INTEGRA study?

Document filter: INTEGRA

Top retrieved chunks:

Rank 1
Chunk ID: PROTO_003_P008_C002
Page: 8
Distance: 0.7177
Text:
According to a study performed by our research group [16], we estimated that each GP would have 15 potential candidate patients with HbA1c > 9%. Assuming that the intra-class correlation coefficient for Primary Care is 0.05 [ 29], resulting in a 1.7 design effect, the required sample size will be 42 * 1.7 = 72 individu

----------------------------------------------------------------------

Rank 2
Chunk ID: PROTO_003_P003_C003
Page: 3
Distance: 0.7989
Text:
treatment will be not considered as exclusion criteria. The exclusion criteria include the following: T2DM controlled by endocrinologist at the moment of inclu- sion, systemic glucocorticoid treatment (ATC code: H02AB) or orlistat treatment (ATC code: A08AB01) (chronic or during the two months prior to inclusion), estimated life expec

## 19. Measure where the known evidence ranks

The OpenAI top results do not immediately show the expected INTEGRA follow-up evidence, but inspecting only the top few results is not enough to diagnose the problem.

We already know from manual document inspection that relevant follow-up information appears on INTEGRA page 6. We will therefore retrieve all INTEGRA chunks and measure exactly where the page 6 chunks rank under both BGE and OpenAI.

This lets us evaluate retrieval using known evidence rather than relying only on visual inspection of the top results.

In [23]:
# Count how many chunks belong to the INTEGRA protocol
# Convert the Pandas/NumPy result into a normal Python integer for Chroma
integra_chunk_count = int(
    chunks_df["document_name"]
    .eq("INTEGRA")
    .sum()
)

print("INTEGRA chunks:", integra_chunk_count)


# Inspect the actual page 6 chunks
integra_page6 = chunks_df[
    (chunks_df["document_name"] == "INTEGRA")
    & (chunks_df["page_number"] == 6)
]

print("\nINTEGRA page 6 chunks:")
print("=" * 70)

for _, row in integra_page6.iterrows():
    text = row["text"].replace("\n", " ")

    # Show the text around "follow" when possible
    follow_position = text.lower().find("follow")

    if follow_position != -1:
        start = max(0, follow_position - 120)
        end = min(len(text), follow_position + 300)
        preview = text[start:end]
    else:
        preview = text[:300]

    print(
        f"\n{row['chunk_id']} | "
        f"{row['token_count']} tokens"
    )
    print(preview)
    print("-" * 70)


# Retrieve every INTEGRA chunk using BGE
all_bge_results = bge_collection.query(
    query_embeddings=[bge_query_embedding.tolist()],
    n_results=integra_chunk_count,
    where={"document_name": "INTEGRA"},
    include=["metadatas", "distances"],
)


# Retrieve every INTEGRA chunk using OpenAI
all_openai_results = openai_collection.query(
    query_embeddings=[openai_query_embedding],
    n_results=integra_chunk_count,
    where={"document_name": "INTEGRA"},
    include=["metadatas", "distances"],
)


# Convert retrieval results into a ranking table
def build_rank_table(results):
    records = []

    for rank, chunk_id in enumerate(
        results["ids"][0],
        start=1,
    ):
        metadata = results["metadatas"][0][rank - 1]
        distance = results["distances"][0][rank - 1]

        records.append(
            {
                "rank": rank,
                "chunk_id": chunk_id,
                "page_number": metadata["page_number"],
                "distance": round(distance, 4),
            }
        )

    return pd.DataFrame(records)


bge_rank_df = build_rank_table(all_bge_results)
openai_rank_df = build_rank_table(all_openai_results)


print("\nBGE — ranks of INTEGRA page 6 chunks:")
print(
    bge_rank_df[
        bge_rank_df["page_number"] == 6
    ].to_string(index=False)
)


print("\nOpenAI — ranks of INTEGRA page 6 chunks:")
print(
    openai_rank_df[
        openai_rank_df["page_number"] == 6
    ].to_string(index=False)
)

INTEGRA chunks: 39

INTEGRA page 6 chunks:

PROTO_003_P006_C001 | 479 tokens
ontains two questions that are scored on a scale from 0 to 4. The sum of the scores of both questions are classified as follows: ≥ 4 total score = “sufficiently” active (encourage the patient to continue their activity) or 0 to 3 total score = “insufficiently” active (encourage the patient to increase their activity). – Patient treatment satisfaction: this is assessed using the Diabetes Treatment Satisfaction Questio
----------------------------------------------------------------------

PROTO_003_P006_C002 | 475 tokens
                                                     of Catalunya Phase 2- Clinical trial Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++   n=72 (Follow up: 0, 3, 12 m) Lleida  Barcelona  Girona   Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++   n=72 (Follow up: 0, 3, 12 m) Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) 

## 20. Inspect lexical matches for the follow-up query

Semantic retrieval did not rank the known follow-up schedule highly enough for reliable Top-K retrieval.

The relevant chunks contain explicit terms such as `Follow up`, so we will check whether simple lexical matching can identify this evidence more directly.

This is a diagnostic step before deciding whether the final retriever should combine semantic and keyword-based search.

In [24]:
# Search INTEGRA chunks for explicit follow-up wording
integra_chunks = chunks_df[
    chunks_df["document_name"] == "INTEGRA"
].copy()


# Count occurrences of the word "follow" in each chunk
integra_chunks["follow_count"] = (
    integra_chunks["text"]
    .str.lower()
    .str.count(r"\bfollow\b")
)


# Keep only chunks containing the lexical term
follow_matches = (
    integra_chunks[
        integra_chunks["follow_count"] > 0
    ]
    .sort_values(
        ["follow_count", "page_number"],
        ascending=[False, True],
    )
)


print("INTEGRA chunks containing 'follow':", len(follow_matches))

print("\nLexical matches:")
print("=" * 70)


for _, row in follow_matches.iterrows():
    text = row["text"].replace("\n", " ")

    follow_position = text.lower().find("follow")

    start = max(0, follow_position - 100)
    end = min(len(text), follow_position + 300)

    print(
        f"\n{row['chunk_id']} | "
        f"Page {row['page_number']} | "
        f"'follow' occurrences: {row['follow_count']}"
    )

    print(text[start:end])
    print("-" * 70)

INTEGRA chunks containing 'follow': 6

Lexical matches:

PROTO_003_P006_C002 | Page 6 | 'follow' occurrences: 6
                                 of Catalunya Phase 2- Clinical trial Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++   n=72 (Follow up: 0, 3, 12 m) Lleida  Barcelona  Girona   Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++   n=72 (Follow up: 0, 3, 12 m) Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++ 
----------------------------------------------------------------------

PROTO_003_P006_C003 | Page 6 | 'follow' occurrences: 3
(Follow up: 0, 3, 12 m) Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++   n=72 (Follow up: 0, 3, 12 m) Control group n= 216 Fig. 1 Study flow chart. *Control group: usual clinical care with the usual control by the family doctor and nurse according to the current CPG p
----------------------------------------------------------

## 21. Build a BM25 lexical retrieval baseline

The follow-up retrieval failure showed that the correct evidence contains strong lexical signals even though pure semantic retrieval ranked it too low.

We will now build a BM25 keyword-search baseline. BM25 ranks chunks according to how well their words match the query. This will later be combined with semantic retrieval to create a hybrid retriever.

In [25]:
import re
from rank_bm25 import BM25Okapi


# Work only with INTEGRA because the document has already been selected
integra_chunks = (
    chunks_df[
        chunks_df["document_name"] == "INTEGRA"
    ]
    .copy()
    .reset_index(drop=True)
)


# Simple tokenizer for lexical retrieval
# It lowercases text and extracts word/number tokens
def lexical_tokenize(text):
    return re.findall(
        r"[A-Za-z0-9]+",
        text.lower(),
    )


# Tokenize every INTEGRA chunk
tokenized_corpus = [
    lexical_tokenize(text)
    for text in integra_chunks["text"]
]


# Build the BM25 search index
bm25 = BM25Okapi(
    tokenized_corpus
)


# Use the same information need as our retrieval test
bm25_query = (
    "participant follow-up schedule "
    "and assessment time points"
)

tokenized_query = lexical_tokenize(
    bm25_query
)


# Calculate a BM25 relevance score for every chunk
bm25_scores = bm25.get_scores(
    tokenized_query
)


# Attach the scores to the chunks
bm25_results = integra_chunks.copy()

bm25_results["bm25_score"] = bm25_scores


# Rank highest-scoring chunks first
bm25_results = (
    bm25_results
    .sort_values(
        "bm25_score",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("Query:")
print(bm25_query)

print("\nTop 5 BM25 results:")
print("=" * 70)


for rank, row in bm25_results.head(5).iterrows():
    text = row["text"].replace("\n", " ")

    follow_position = text.lower().find("follow")

    if follow_position != -1:
        start = max(
            0,
            follow_position - 100,
        )
        end = min(
            len(text),
            follow_position + 300,
        )

        preview = text[start:end]
    else:
        preview = text[:300]

    print(f"\nRank {rank + 1}")
    print("Chunk ID:", row["chunk_id"])
    print("Page:", row["page_number"])
    print(
        "BM25 score:",
        round(row["bm25_score"], 4),
    )
    print("Text:")
    print(preview)

    print("\n" + "-" * 70)

Query:
participant follow-up schedule and assessment time points

Top 5 BM25 results:

Rank 1
Chunk ID: PROTO_003_P006_C002
Page: 6
BM25 score: 8.8469
Text:
                                 of Catalunya Phase 2- Clinical trial Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++   n=72 (Follow up: 0, 3, 12 m) Lleida  Barcelona  Girona   Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++   n=72 (Follow up: 0, 3, 12 m) Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++ 

----------------------------------------------------------------------

Rank 2
Chunk ID: PROTO_003_P005_C002
Page: 5
BM25 score: 8.2905
Text:
ups will be extracted from the SIDIAP database for 12 months for each patient. Participants will be followed up until they experience the outcomes of interest, die, leave the SIDIAP database (e.g., change of address) or complete the follow-up (31 October 2018). If the inter- vention groups reach th

## 22. Combine semantic and lexical retrieval

BM25 ranked the known INTEGRA follow-up evidence first, while semantic retrieval ranked the same evidence much lower.

This suggests that the two retrieval methods capture different useful signals. Semantic search captures similarity in meaning, while BM25 is strong when important protocol terminology appears explicitly in the text.

We will combine OpenAI semantic retrieval and BM25 using Reciprocal Rank Fusion. This method combines ranking positions rather than incompatible raw similarity scores.

In [26]:
# Use the information need only because INTEGRA is already selected by metadata
hybrid_query = (
    "participant follow-up schedule "
    "and assessment time points"
)


# Create an OpenAI embedding for the focused query
hybrid_query_response = openai_client.embeddings.create(
    model=openai_embedding_model,
    input=hybrid_query,
)

hybrid_query_embedding = (
    hybrid_query_response.data[0].embedding
)


# Retrieve all INTEGRA chunks using OpenAI semantic search
semantic_results = openai_collection.query(
    query_embeddings=[hybrid_query_embedding],
    n_results=integra_chunk_count,
    where={"document_name": "INTEGRA"},
    include=["metadatas", "distances"],
)


# Record the semantic rank of every returned chunk
semantic_rank = {
    chunk_id: rank
    for rank, chunk_id in enumerate(
        semantic_results["ids"][0],
        start=1,
    )
}


# Record the BM25 rank of every INTEGRA chunk
bm25_rank = {
    chunk_id: rank
    for rank, chunk_id in enumerate(
        bm25_results["chunk_id"],
        start=1,
    )
}


# Reciprocal Rank Fusion combines ranks instead of raw scores
rrf_k = 60

hybrid_records = []

for _, row in integra_chunks.iterrows():
    chunk_id = row["chunk_id"]

    semantic_position = semantic_rank[chunk_id]
    bm25_position = bm25_rank[chunk_id]

    rrf_score = (
        1 / (rrf_k + semantic_position)
        + 1 / (rrf_k + bm25_position)
    )

    hybrid_records.append(
        {
            "chunk_id": chunk_id,
            "page_number": row["page_number"],
            "semantic_rank": semantic_position,
            "bm25_rank": bm25_position,
            "rrf_score": rrf_score,
            "text": row["text"],
        }
    )


# Rank chunks by the combined RRF score
hybrid_results = (
    pd.DataFrame(hybrid_records)
    .sort_values(
        "rrf_score",
        ascending=False,
    )
    .reset_index(drop=True)
)


print("Query:")
print(hybrid_query)

print("\nTop 5 hybrid results:")
print("=" * 70)


for rank, row in hybrid_results.head(5).iterrows():
    text = row["text"].replace("\n", " ")

    follow_position = text.lower().find("follow")

    if follow_position != -1:
        start = max(0, follow_position - 100)
        end = min(len(text), follow_position + 300)
        preview = text[start:end]
    else:
        preview = text[:300]

    print(f"\nRank {rank + 1}")
    print("Chunk ID:", row["chunk_id"])
    print("Page:", row["page_number"])
    print("Semantic rank:", row["semantic_rank"])
    print("BM25 rank:", row["bm25_rank"])
    print("RRF score:", round(row["rrf_score"], 6))
    print("Text:")
    print(preview)

    print("\n" + "-" * 70)

Query:
participant follow-up schedule and assessment time points

Top 5 hybrid results:

Rank 1
Chunk ID: PROTO_003_P006_C003
Page: 6
Semantic rank: 2
BM25 rank: 4
RRF score: 0.031754
Text:
(Follow up: 0, 3, 12 m) Intervention group 1+   n=72 (Follow up:  0, 3, 6, 12 m) Intervention group 2++   n=72 (Follow up: 0, 3, 12 m) Control group n= 216 Fig. 1 Study flow chart. *Control group: usual clinical care with the usual control by the family doctor and nurse according to the current CPG p

----------------------------------------------------------------------

Rank 2
Chunk ID: PROTO_003_P005_C002
Page: 5
Semantic rank: 6
BM25 rank: 2
RRF score: 0.031281
Text:
ups will be extracted from the SIDIAP database for 12 months for each patient. Participants will be followed up until they experience the outcomes of interest, die, leave the SIDIAP database (e.g., change of address) or complete the follow-up (31 October 2018). If the inter- vention groups reach the study sample earlier than 31 July

## 23. Identify evidence for retrieval test questions

The hybrid retriever successfully recovered the known INTEGRA follow-up evidence, but one successful query is not enough to evaluate a retrieval strategy.

We will inspect the corpus for common protocol concepts such as eligibility, exclusion criteria, outcomes, randomisation and follow-up. These evidence locations will be used to create a small set of retrieval questions with known source documents and pages.

In [27]:
# Terms commonly associated with useful protocol questions
test_terms = [
    "inclusion criteria",
    "exclusion criteria",
    "primary outcome",
    "secondary outcome",
    "random",
    "follow",
    "sample size",
]


# Show which documents and pages contain each term
for term in test_terms:
    matches = chunks_df[
        chunks_df["text"]
        .str.lower()
        .str.contains(
            term,
            regex=False,
            na=False,
        )
    ]

    evidence_locations = (
        matches[
            [
                "document_name",
                "page_number",
                "chunk_id",
            ]
        ]
        .drop_duplicates()
        .head(8)
    )

    print(f"\nTERM: {term}")
    print("=" * 60)

    if evidence_locations.empty:
        print("No direct matches found.")
    else:
        print(
            evidence_locations.to_string(
                index=False
            )
        )


TERM: inclusion criteria
document_name  page_number            chunk_id
      CERTAIN            1 PROTO_001_P001_C003
      CERTAIN            1 PROTO_001_P001_C004
  CARE_STROKE            1 PROTO_002_P001_C003
  CARE_STROKE            1 PROTO_002_P001_C004
  CARE_STROKE            3 PROTO_002_P003_C001
      INTEGRA            3 PROTO_003_P003_C002
      INTEGRA            4 PROTO_003_P004_C002
      INTEGRA            5 PROTO_003_P005_C002

TERM: exclusion criteria
document_name  page_number            chunk_id
      INTEGRA            3 PROTO_003_P003_C002
      INTEGRA            3 PROTO_003_P003_C003
      INTEGRA            7 PROTO_003_P007_C001
       THP_TA            3 PROTO_005_P003_C003

TERM: primary outcome
document_name  page_number            chunk_id
      CERTAIN            4 PROTO_001_P004_C002
      CERTAIN            4 PROTO_001_P004_C003
  CARE_STROKE            1 PROTO_002_P001_C002
  CARE_STROKE            2 PROTO_002_P002_C003
  CARE_STROKE            4 PROTO

## 24. Summarise candidate evidence locations

The corpus contains multiple direct matches for common protocol concepts, but the previous output was too long to inspect comfortably.

We will create a compact summary showing which documents contain each concept, the first matching page, and how many chunks contain the term. This will help us choose a balanced retrieval test set across all five protocols.

In [28]:
# Build a compact summary of direct lexical evidence
evidence_summary_records = []


for term in test_terms:
    matches = chunks_df[
        chunks_df["text"]
        .str.lower()
        .str.contains(
            term,
            regex=False,
            na=False,
        )
    ]

    # Summarise matches separately for each document
    for document_name, group in matches.groupby("document_name"):
        evidence_summary_records.append(
            {
                "term": term,
                "document_name": document_name,
                "first_page": int(group["page_number"].min()),
                "matching_chunks": len(group),
            }
        )


evidence_summary_df = (
    pd.DataFrame(evidence_summary_records)
    .sort_values(
        ["term", "document_name"]
    )
    .reset_index(drop=True)
)


print(evidence_summary_df.to_string(index=False))

              term document_name  first_page  matching_chunks
exclusion criteria       INTEGRA           3                3
exclusion criteria        THP_TA           3                1
            follow   CARE_STROKE           1               13
            follow       CERTAIN           1               17
            follow       INTEGRA           2               13
            follow        LISTEN           1               16
            follow        THP_TA           6                9
inclusion criteria   CARE_STROKE           1                3
inclusion criteria       CERTAIN           1                2
inclusion criteria       INTEGRA           3                4
inclusion criteria        THP_TA           3                1
   primary outcome   CARE_STROKE           1                4
   primary outcome       CERTAIN           4                2
   primary outcome        LISTEN           7                3
   primary outcome        THP_TA           1                9
        

## 24. Summarise candidate evidence locations

The corpus contains multiple direct matches for common protocol concepts, but the previous output was too long to inspect comfortably.

We will create a compact summary showing which documents contain each concept, the first matching page, and how many chunks contain the term. This will help us choose a balanced retrieval test set across all five protocols.

In [29]:
# Build a compact summary of direct lexical evidence
evidence_summary_records = []


for term in test_terms:
    matches = chunks_df[
        chunks_df["text"]
        .str.lower()
        .str.contains(
            term,
            regex=False,
            na=False,
        )
    ]

    # Summarise matches separately for each document
    for document_name, group in matches.groupby("document_name"):
        evidence_summary_records.append(
            {
                "term": term,
                "document_name": document_name,
                "first_page": int(group["page_number"].min()),
                "matching_chunks": len(group),
            }
        )


evidence_summary_df = (
    pd.DataFrame(evidence_summary_records)
    .sort_values(
        ["term", "document_name"]
    )
    .reset_index(drop=True)
)


print(evidence_summary_df.to_string(index=False))

              term document_name  first_page  matching_chunks
exclusion criteria       INTEGRA           3                3
exclusion criteria        THP_TA           3                1
            follow   CARE_STROKE           1               13
            follow       CERTAIN           1               17
            follow       INTEGRA           2               13
            follow        LISTEN           1               16
            follow        THP_TA           6                9
inclusion criteria   CARE_STROKE           1                3
inclusion criteria       CERTAIN           1                2
inclusion criteria       INTEGRA           3                4
inclusion criteria        THP_TA           3                1
   primary outcome   CARE_STROKE           1                4
   primary outcome       CERTAIN           4                2
   primary outcome        LISTEN           7                3
   primary outcome        THP_TA           1                9
        

## 25. Create a small retrieval diagnostic set

The corpus contains enough evidence across all five protocols to test retrieval more broadly.

We will create a small set of ten diagnostic questions covering different protocol concepts and source documents. This is a Day 3 smoke test rather than the final evaluation dataset. The larger and more carefully labelled evaluation set will be built later.

In [30]:
# Create a small balanced retrieval test set
diagnostic_questions = pd.DataFrame(
    [
        {
            "question_id": "Q01",
            "document_name": "CERTAIN",
            "query": "What is the primary outcome of the study?",
            "evidence_term": "primary outcome",
        },
        {
            "question_id": "Q02",
            "document_name": "CERTAIN",
            "query": "How was the study sample size determined?",
            "evidence_term": "sample size",
        },
        {
            "question_id": "Q03",
            "document_name": "CARE_STROKE",
            "query": "What are the inclusion criteria for participants?",
            "evidence_term": "inclusion criteria",
        },
        {
            "question_id": "Q04",
            "document_name": "CARE_STROKE",
            "query": "What is the primary outcome of the study?",
            "evidence_term": "primary outcome",
        },
        {
            "question_id": "Q05",
            "document_name": "INTEGRA",
            "query": "What are the exclusion criteria?",
            "evidence_term": "exclusion criteria",
        },
        {
            "question_id": "Q06",
            "document_name": "INTEGRA",
            "query": "What are the participant follow-up time points?",
            "evidence_term": "follow",
        },
        {
            "question_id": "Q07",
            "document_name": "LISTEN",
            "query": "What is the primary outcome of the study?",
            "evidence_term": "primary outcome",
        },
        {
            "question_id": "Q08",
            "document_name": "LISTEN",
            "query": "How was the study sample size determined?",
            "evidence_term": "sample size",
        },
        {
            "question_id": "Q09",
            "document_name": "THP_TA",
            "query": "What are the exclusion criteria?",
            "evidence_term": "exclusion criteria",
        },
        {
            "question_id": "Q10",
            "document_name": "THP_TA",
            "query": "How was the study sample size determined?",
            "evidence_term": "sample size",
        },
    ]
)


print("Diagnostic questions:", len(diagnostic_questions))
print(
    diagnostic_questions[
        [
            "question_id",
            "document_name",
            "query",
            "evidence_term",
        ]
    ].to_string(index=False)
)

Diagnostic questions: 10
question_id document_name                                             query      evidence_term
        Q01       CERTAIN         What is the primary outcome of the study?    primary outcome
        Q02       CERTAIN         How was the study sample size determined?        sample size
        Q03   CARE_STROKE What are the inclusion criteria for participants? inclusion criteria
        Q04   CARE_STROKE         What is the primary outcome of the study?    primary outcome
        Q05       INTEGRA                  What are the exclusion criteria? exclusion criteria
        Q06       INTEGRA   What are the participant follow-up time points?             follow
        Q07        LISTEN         What is the primary outcome of the study?    primary outcome
        Q08        LISTEN         How was the study sample size determined?        sample size
        Q09        THP_TA                  What are the exclusion criteria? exclusion criteria
        Q10        THP_TA

## 26. Compare semantic, BM25 and hybrid retrieval

We will now evaluate the three retrieval strategies on the same ten diagnostic questions across all five protocols.

For this Day 3 smoke test, a retrieval is counted as a Top-3 hit when at least one of the first three retrieved chunks contains the expected evidence term. This is only a lightweight diagnostic metric; the final evaluation will use manually labelled evidence pages and chunks rather than simple term matching.

The comparison will help us choose the retrieval strategy to carry forward into the grounded RAG system.

In [ ]:
# Store the result for every diagnostic question
diagnostic_results = []


for _, question_row in diagnostic_questions.iterrows():

    question_id = question_row["question_id"]
    document_name = question_row["document_name"]
    query = question_row["query"]
    evidence_term = question_row["evidence_term"]

    # Work only with chunks from the requested protocol
    document_chunks = (
        chunks_df[
            chunks_df["document_name"] == document_name
        ]
        .copy()
        .reset_index(drop=True)
    )

    document_chunk_count = len(document_chunks)


    # Identify chunks containing the expected evidence term
    evidence_chunk_ids = set(
        document_chunks[
            document_chunks["text"]
            .str.lower()
            .str.contains(
                evidence_term,
                regex=False,
                na=False,
            )
        ]["chunk_id"]
    )

    # 1. OpenAI semantic retrieval

    query_response = openai_client.embeddings.create(
        model=openai_embedding_model,
        input=query,
    )

    query_embedding = query_response.data[0].embedding

    semantic_results = openai_collection.query(
        query_embeddings=[query_embedding],
        n_results=document_chunk_count,
        where={
            "document_name": document_name
        },
        include=["metadatas"],
    )

    semantic_ids = semantic_results["ids"][0]

    semantic_rank = {
        chunk_id: rank
        for rank, chunk_id in enumerate(
            semantic_ids,
            start=1,
        )
    }


    # 2. BM25 lexical retrieval

    tokenized_document = [
        lexical_tokenize(text)
        for text in document_chunks["text"]
    ]

    document_bm25 = BM25Okapi(
        tokenized_document
    )

    query_tokens = lexical_tokenize(query)

    bm25_scores = document_bm25.get_scores(
        query_tokens
    )

    bm25_order = (
        pd.DataFrame(
            {
                "chunk_id": document_chunks["chunk_id"],
                "bm25_score": bm25_scores,
            }
        )
        .sort_values(
            "bm25_score",
            ascending=False,
        )
        ["chunk_id"]
        .tolist()
    )

    bm25_rank = {
        chunk_id: rank
        for rank, chunk_id in enumerate(
            bm25_order,
            start=1,
        )
    }


    # 3. Hybrid retrieval using Reciprocal Rank Fusion

    hybrid_records = []

    for chunk_id in document_chunks["chunk_id"]:

        rrf_score = (
            1 / (60 + semantic_rank[chunk_id])
            + 1 / (60 + bm25_rank[chunk_id])
        )

        hybrid_records.append(
            {
                "chunk_id": chunk_id,
                "rrf_score": rrf_score,
            }
        )

    hybrid_ids = (
        pd.DataFrame(hybrid_records)
        .sort_values(
            "rrf_score",
            ascending=False,
        )
        ["chunk_id"]
        .tolist()
    )


  
    # Check whether expected evidence reaches Top 3

    semantic_top3 = semantic_ids[:3]
    bm25_top3 = bm25_order[:3]
    hybrid_top3 = hybrid_ids[:3]

    semantic_hit3 = bool(
        evidence_chunk_ids.intersection(
            semantic_top3
        )
    )

    bm25_hit3 = bool(
        evidence_chunk_ids.intersection(
            bm25_top3
        )
    )

    hybrid_hit3 = bool(
        evidence_chunk_ids.intersection(
            hybrid_top3
        )
    )


    # Find the best rank of any expected evidence chunk
    semantic_best_rank = min(
        semantic_rank[chunk_id]
        for chunk_id in evidence_chunk_ids
    )

    bm25_best_rank = min(
        bm25_rank[chunk_id]
        for chunk_id in evidence_chunk_ids
    )

    hybrid_rank = {
        chunk_id: rank
        for rank, chunk_id in enumerate(
            hybrid_ids,
            start=1,
        )
    }

    hybrid_best_rank = min(
        hybrid_rank[chunk_id]
        for chunk_id in evidence_chunk_ids
    )


    diagnostic_results.append(
        {
            "question_id": question_id,
            "document": document_name,
            "semantic_rank": semantic_best_rank,
            "bm25_rank": bm25_best_rank,
            "hybrid_rank": hybrid_best_rank,
            "semantic_hit3": semantic_hit3,
            "bm25_hit3": bm25_hit3,
            "hybrid_hit3": hybrid_hit3,
        }
    )


# Convert results into a dataframe
diagnostic_results_df = pd.DataFrame(
    diagnostic_results
)


print("Per-question results:")
print(
    diagnostic_results_df.to_string(
        index=False
    )
)


print("\nTop-3 retrieval summary:")
print("=" * 50)

print(
    "OpenAI semantic:",
    diagnostic_results_df["semantic_hit3"].sum(),
    "/",
    len(diagnostic_results_df),
)

print(
    "BM25:",
    diagnostic_results_df["bm25_hit3"].sum(),
    "/",
    len(diagnostic_results_df),
)

print(
    "Hybrid:",
    diagnostic_results_df["hybrid_hit3"].sum(),
    "/",
    len(diagnostic_results_df),
)

Per-question results:
question_id    document  semantic_rank  bm25_rank  hybrid_rank  semantic_hit3  bm25_hit3  hybrid_hit3
        Q01     CERTAIN              2          8            3           True      False         True
        Q02     CERTAIN              1          1            1           True       True         True
        Q03 CARE_STROKE              1          1            1           True       True         True
        Q04 CARE_STROKE              1          1            1           True       True         True
        Q05     INTEGRA              1          1            1           True       True         True
        Q06     INTEGRA              1          1            1           True       True         True
        Q07      LISTEN              2          4            3           True      False         True
        Q08      LISTEN              2          1            1           True       True         True
        Q09      THP_TA              1          1           

## Day 3 Findings

The 172 validated protocol chunks from Day 2 were loaded and extended with deterministic SHA-256 text fingerprints and readable preprocessing metadata.

Semantic retrieval was first explored using `all-MiniLM-L6-v2`. Although the model successfully produced 384-dimensional embeddings for all chunks, retrieval inspection revealed a model/chunk compatibility problem. MiniLM has a maximum sequence length of 256 tokens, and measurement using its own tokenizer showed that 145 of 172 chunks (84.3%) exceeded this limit.

A longer-context local embedding model, `BAAI/bge-small-en-v1.5`, was therefore evaluated. It supports a 512-token input length, and all 172 chunks fitted within its limit. A separate persistent Chroma index was created using the BGE embeddings.

OpenAI's `text-embedding-3-small` was also evaluated as a hosted semantic-retrieval model. All 172 chunks were embedded successfully into 1536-dimensional vectors and stored in a separate Chroma collection.

Manual inspection of an INTEGRA follow-up question showed that pure semantic retrieval could rank known answer-bearing chunks too low, particularly where PDF extraction had converted a flow chart into irregular table-like text.

Lexical retrieval was therefore tested using BM25. For the INTEGRA follow-up failure case, BM25 ranked the primary answer-bearing chunk on page 6 first.

OpenAI semantic retrieval and BM25 were then combined using Reciprocal Rank Fusion (RRF). The hybrid retriever successfully moved answer-bearing INTEGRA page 6 evidence to the top of the ranking.

A ten-question diagnostic set covering all five protocols was used as a Day 3 smoke test. Using a simple evidence-term Top-3 criterion, OpenAI semantic retrieval achieved 10/10 hits, BM25 achieved 8/10, and the hybrid retriever achieved 10/10.

These results are diagnostic rather than final evaluation metrics. The evidence labels are currently based on direct evidence terms rather than manually verified gold-standard chunks. A larger evaluation set with exact expected evidence and unsupported questions will be used later.

The retrieval architecture carried forward is metadata-filtered OpenAI semantic retrieval combined with BM25 lexical retrieval using RRF. BGE remains a reproducible local baseline, while the MiniLM experiment documents an important embedding-context compatibility failure discovered during development.